In [170]:
%load_ext autoreload
%autoreload 2
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2
from scipy.stats import norm
from scipy import optimize
from numpy.random import default_rng


import w8_estimation as est 
import w8_LinearModel as lm
import w8_probit as probit
import w8_logit as logit

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [171]:
dat = pd.read_csv('ppcs_cc.csv')
df=dat.copy()

df.drop(columns=['osplit', 'year'], inplace=True) #no variance in these variables, so we drop them
df = pd.get_dummies(df, columns=['sincome', 'spop', 'inctype_lin'], dtype=int) #make dummy variables for categorical variables

#add age squared variable
df['sage_sq'] = df['sage']**2
df['omajnonwhite'] = 1 - df['omajwhite'] #create a variable for non-white officers
df['const'] = 1

#choose reference categories for categorical variables
df = df.drop(columns=['swhite', 'omajwhite', 'sincome_2', 'spop_2', 'inctype_lin_2', 'omajhisp', 'omajother', 'omajblack']) #drop reference categories

df.insert(0, 'const', df.pop('const'))


# Create different data-frames for different models
x1 = ['const', 'sblack', 'shisp', 'sother']
x2 = x1 + ['sage', 'sage_sq', 'smale', 'sempl', 'sincome_1', 'sincome_3',
           'spop_1', 'spop_3', 'spop_4','inctype_lin_1']
x3 = x2 + ['daytime', 'omajnonwhite']
x4 = x3 + ['sbehavior']

# Final check
len(df.columns) == len(x4) + 1 # +1 for the dependent variable

y = df.anyuseofforce_coded.values

models = {"x1": x1, "x2": x2, "x3": x3, "x4": x4}




In [172]:
def estimate_and_compute_marg_effs(y, x, white, black, hisp): 
    '''Estimate and compute APES
    Args. 
        y: N-vector of discrete outcomes
        x: N*J matrix of explanatory variables
        white: N*J matrix of counterfactual variables used to predict avg. propensity for whites 
        black: N*J matrix of counterfactual variables used to predict avg. propensity for blacks 
        hisp: N*J matrix of counterfactual variables used to predict avg. propensity for hispanics 
        
    Returns
        Average partial effects for black v.s white and hisp v.s whites for probit and logit.
    '''
    assert y.ndim == 1 , 'y must be a flattened array'
    assert x.ndim == 2 , 'x must be 2-dimensional'
    
    N,K = x.shape

    # Check full rank
    eigenvalues = np.linalg.eigvals(x.T@x)
    all_pos = np.all(eigenvalues > 0)

    #try:
    # 1. Starting values
    theta_0_probit = probit.starting_values(y=y, x=x)
    theta_0_logit = logit.starting_values(y=y, x=x)
        
    # 2. Estimate probit and logit models
    probit_res =  est.estimate(probit.q, theta_0_probit, y=y, x=x, method='BFGS', options={'disp':False}) # Fill in: estimate parameters
    logit_res = est.estimate(logit.q, theta_0_logit, y=y, x=x, method='BFGS', options={'disp':False})
        
    betahat_probit = probit_res['theta'].reshape(-1, 1)    # More accurate parameter estimates
    betahat_logit = logit_res['theta'].reshape(-1, 1)      # More accurate parameter estimates

            
            
    # 2. Vector of marginal effects (Black v.s white)
    PE_black_white_probit_vec = (
            probit.predict(betahat_probit, black) - probit.predict(betahat_probit, white))
    PE_black_white_logit_vec = (
            logit.predict(betahat_logit, black) - logit.predict(betahat_logit, white))
            
    # 2.1 Vector of marginal effects (Black v.s white)
    PE_hisp_white_probit_vec = (
                probit.predict(betahat_probit, hisp) - probit.predict(betahat_probit, white))
    PE_hisp_white_logit_vec = (
                logit.predict(betahat_logit, hisp) - logit.predict(betahat_logit, white))
        
            
    # 3. Average partial effects
    APE_BW_probit = PE_black_white_probit_vec.mean()
    APE_BW_logit = PE_black_white_logit_vec.mean()
        
    APE_HW_probit = PE_hisp_white_probit_vec.mean()
    APE_HW_logit = PE_hisp_white_logit_vec.mean()
    

    # 4. return 
    return PE_black_white_probit_vec, PE_black_white_logit_vec, PE_hisp_white_probit_vec, PE_hisp_white_logit_vec
        
    #except:
        #print('ERORR OCCURED IN ESTIMATION PROCEDURE')
        #print(eigenvalues)
        #return np.nan, np.nan, np.nan, np.nan


def compute_APE(vec):
    return vec.mean()
    



In [173]:
def bootstrap_sample(y, df, columns, test=False): 
    '''bootstrap_sample: samples a new dataset (with replacement) from the input. 
    Args. 
        y: 1-dimensional N-array
        df: Dataframe wil all relevant variables
        columns: The model used (x1, x2, x3, x4) -> List of variables 
    Returns
        tuple: y_i, x_i 
            y_i: N-array
            x_i: (N,K) matrix 
            white: (N,K) matrix of x_i but with all whites
            black: (N,K) matrix of x_i but with all blacks
            hisp: (N,K) matrix of x_1 but with all hisp
    '''
    N = y.size

    x = df[columns].to_numpy()

    # FILL IN 
    if test==False:
        ii_boot = np.random.choice(N, size=N, replace=True) # vector of indices for rows (Check if size is correct.. I think yes)
    else:
        ii_boot = np.arange(N)                  # Used to generate the original data (no bootstrap).

    y_i = y[ii_boot] # selection of N rows from y 
    x_i = x[ii_boot] # selection of N rows from x 

    white = x.copy()       # Convert df to numpy
    counterfactual = {"sblack": 0, "shisp": 0, "sother": 0} #White
    for variable, value in counterfactual.items():
        white[:,columns.index(variable)] = value        # Set all race dummies to zero to make a white counterfactual.

    white = white[ii_boot]   # Select only bootstrap sample
    
    # blacks
    black = white.copy()
    black[:, columns.index("sblack")] = 1               # Make a black counterfactual by setting sblack = 1 for all obs
        
    # Hispanics
    hisp = white.copy()
    hisp[:, columns.index("shisp")] = 1                 # Make a hispanic counterfactual by setting shisp = 1 for all obs
        
    # Check that everything works as it should
    assert white[:,1].sum()==0 and white[:,2].sum()==0 and white[:,3].sum()==0
    assert black[:,columns.index("sblack")].sum()==N and black[:,2].sum()==0 and black[:,3].sum()==0
    assert hisp[:,1].sum()==0 and hisp[:,columns.index("shisp")].sum()==N and hisp[:,3].sum()==0


    return y_i, x_i, white, black, hisp 

In [174]:
def has_separation(y_b, x_b):
    """Return the index j of the first offending column, or None if no problem is found."""
    for j in range(x_b.shape[1]):
        vals = np.unique(x_b[:, j])
        if vals.size == 1:
            if j == 0: continue            # the intercept
            return j                       # empty/full category
        if vals.size == 2:                 # binary column
            for v in vals:
                m = x_b[:, j] == v
                if y_b[m].sum() == 0 or y_b[m].sum() == m.sum():
                    return j
    return None

In [183]:
## Testing the function. Correct output is 0.001897 for logit APE for black
y_test, x_test, white_test, black_test, hisp_test = bootstrap_sample(y=y,df=df, columns=x2, test=True)
t = has_separation(y_test, x_test)
print(f'Has seperation for some dummy: {t}')
pe_probit_BW, pe_logit_BW, pe_probit_HW, pe_logit_HW = estimate_and_compute_marg_effs(y=y_test, x=x_test, white=white_test, black=black_test, hisp=hisp_test)
ape_probit_BW = compute_APE(pe_probit_BW)
ape_logit_BW = compute_APE(pe_logit_BW)
ape_probit_HW = compute_APE(pe_probit_HW)
ape_logit_HW = compute_APE(pe_logit_HW)

print(ape_probit_BW)
print(ape_logit_BW)
print(ape_probit_HW)
print(ape_logit_HW)

Has seperation for some dummy: None


/Users/mathiaslindholmrasmussen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Semester 9/Advanced Microeconometrics/Adv_microeconometrics_assignments/project 3/w8_logit.py:14: RuntimeWarning: overflow encountered in exp
  Gz = 1. / (1. + np.exp(-z))


0.0009256280160566022
0.001919001540911315
0.0058048637752771675
0.005833667835246476


In [ ]:
APE_black_white_rows = []       # STORE ALL BLACK V.S WHITE RESULTS
## Hispanic V.S White
APE_hisp_white_rows = []       # STORE ALL HISP V.S WHITE RESULTS

for i, (name, columns) in enumerate(models.items()):
    # TRUE APEs for all models
    y_, x_, white_, black_, hisp_ = bootstrap_sample(y=y,df=df, columns=columns, test=True)
    pe_probit_bw, pe_logit_bw, pe_probit_hw, pe_logit_hw = estimate_and_compute_marg_effs(y=y_, x=x_, white=white_, black=black_, hisp=hisp_)
    ape_probit_BW = compute_APE(pe_probit_bw)
    ape_logit_BW = compute_APE(pe_logit_bw)
    ape_probit_HW = compute_APE(pe_probit_hw)
    ape_logit_HW = compute_APE(pe_logit_hw)


    # Subpopulations (Black v.s white counterfactual)
    blacks_dummy = x_[:, columns.index('sblack')]       # A 0/1 vector indicating if person is black or not
    N_blacks = blacks_dummy.sum()
    ape_probit_BW_blacks = compute_APE(pe_probit_bw[blacks_dummy==1])
    ape_logit_BW_blacks = compute_APE(pe_logit_bw[blacks_dummy==1])

    whites_dummy = x_[:, [columns.index('sblack'), columns.index('shisp'), columns.index('sother')]].sum(axis=1) # A 0/1 vector indicating if person not white
    N_whites = (whites_dummy==0).sum()
    ape_probit_BW_whites = compute_APE(pe_probit_bw[whites_dummy==0])
    ape_logit_BW_whites = compute_APE(pe_logit_bw[whites_dummy==0])

    # Subpopulations (Hispanic v.s white counterfactual)
    hisp_dummy = x_[:, columns.index('shisp')]      # A 0/1 vector indicating if person is black or not
    N_hisps = hisp_dummy.sum()
    ape_probit_HW_hisps = compute_APE(pe_probit_hw[hisp_dummy==1])
    ape_logit_HW_hisps = compute_APE(pe_logit_hw[hisp_dummy==1])

    ape_probit_HW_whites = compute_APE(pe_probit_hw[whites_dummy==0])
    ape_logit_HW_whites = compute_APE(pe_logit_hw[whites_dummy==0])



    APE_black_white_rows.append({
        "Model": name,
        "Probit APE": ape_probit_BW,
        "Logit APE": ape_logit_BW,
        "Probit APE (black sub)": ape_probit_BW_blacks,
        "Logit APE (black sub)": ape_logit_BW_blacks,
        "Probit APE (white sub)": ape_probit_BW_whites,
        "Logit APE (white sub)": ape_logit_BW_whites
    })

    APE_hisp_white_rows.append({
            "Model": name,
            "Probit APE": ape_probit_HW,
            "Logit APE": ape_logit_HW,
            "Probit APE (hisp sub)": ape_probit_HW_hisps,
            "Logit APE (hisp sub)": ape_logit_HW_hisps,
            "Probit APE (white sub)": ape_probit_HW_whites,
            "Logit APE (white sub)": ape_logit_HW_whites
        })


######## BOOTSTRAP TO GET STD. ERRORS ###########
boot=10                         # Should be changed later
# Set seed for random sampling.
seed = 42
rng = default_rng(seed)


apes_prob_boot_bw = np.empty((boot, 4))
apes_logit_boot_bw = np.empty((boot, 4))
apes_prob_boot_hw = np.empty((boot, 4))
apes_logit_boot_hw = np.empty((boot, 4))

variable_failures = []

# subpopulations
apes_prob_boot_bw_blacks = np.empty((boot, 4))      # APE: black v.s white (Black subpopulation), probit
apes_logit_boot_bw_blacks = np.empty((boot, 4))     # APE: black v.s. white (Black subpopulation), logit
apes_prob_boot_bw_whites = np.empty((boot, 4))      # APE: black v.s. white (White subpopulation), probit
apes_logit_boot_bw_whites = np.empty((boot, 4))     # APE: black v.s. white (White subpopultion), logit

apes_prob_boot_hw_hisps = np.empty((boot, 4))       # APE: hisp v.s white (hisp sub), probit
apes_logit_boot_hw_hisps = np.empty((boot, 4))      # APE: hisp v.s white (hisp sub), logit
apes_prob_boot_hw_whites = np.empty((boot, 4))      # APE: hisp v.s white (white sub), probit
apes_logit_boot_hw_whites = np.empty((boot, 4))     # APE: hist v.s white (white sub), logit


# Bootstraps for SE (One bootstrap draw for all models)
for t in range(boot):
    print(f'Bootstrap {t+1}')
    y_boot, x_boot, white_boot, black_boot, hisp_boot = bootstrap_sample(y=y,df=df, columns=x4, test=False)    # Draws a sample using all columns
    
    # Loop over all models
    for i, (name, columns) in enumerate(models.items()):       
        x_boot_ = x_boot[:, 0:len(columns)]
        white_boot_ = white_boot[:, 0:len(columns)]
        black_boot_ = black_boot[:, 0:len(columns)]
        hisp_boot_ = hisp_boot[:, 0:len(columns)]
        
        test = has_separation(y_boot, x_boot_)
        
        if test != None:
            print('Seperation failure')
            print(f'Variable {columns[test]} in model {name}')
            variable_failures.append(columns[test])
            
            apes_prob_boot_bw[t, i] =  np.nan
            apes_logit_boot_bw[t, i] = np.nan
            apes_prob_boot_hw[t, i] = np.nan
            apes_logit_boot_hw[t, i] = np.nan

            apes_prob_boot_bw_blacks[t, i] = np.nan      # APE: black v.s white (Black subpopulation), probit
            apes_logit_boot_bw_blacks[t, i] = np.nan     # APE: black v.s. white (Black subpopulation), logit
            apes_prob_boot_bw_whites[t, i] = np.nan      # APE: black v.s. white (White subpopulation), probit
            apes_logit_boot_bw_whites[t, i] = np.nan     # APE: black v.s. white (White subpopultion), logit

            apes_prob_boot_hw_hisps[t, i] = np.nan       # APE: hisp v.s white (hisp sub), probit
            apes_logit_boot_hw_hisps[t, i] = np.nan      # APE: hisp v.s white (hisp sub), logit
            apes_prob_boot_hw_whites[t, i] = np.nan      # APE: hisp v.s white (white sub), probit
            apes_logit_boot_hw_whites[t, i] = np.nan     # APE: hist v.s white (white sub), logit


        else:
            pe_prob_bw_boot, pe_logit_bw_boot, pe_probit_hw_boot, pe_logit_hw_boot = estimate_and_compute_marg_effs(y=y_boot, x=x_boot_, white=white_boot_, black=black_boot_, hisp=hisp_boot_)
            
            apes_prob_boot_bw[t, i] =  compute_APE(pe_prob_bw_boot)
            apes_logit_boot_bw[t, i] = compute_APE(pe_logit_bw_boot)
            apes_prob_boot_hw[t, i] = compute_APE(pe_probit_hw_boot)
            apes_logit_boot_hw[t, i] = compute_APE(pe_logit_hw_boot)
            
            # subpopulations
            blacks_dummy = x_boot_[:, columns.index('sblack')]
            whites_dummy = x_boot_[:, [columns.index('sblack'), columns.index('shisp'), columns.index('sother')]].sum(axis=1)
            hisp_dummy = x_boot_[:, columns.index('shisp')]
            
            # Calculate APEs for subpopulations
            apes_prob_boot_bw_blacks[t,i] = compute_APE(pe_prob_bw_boot[blacks_dummy==1])     # APE: black v.s white (Black subpopulation), probit
            apes_logit_boot_bw_blacks[t,i] = compute_APE(pe_logit_bw_boot[blacks_dummy==1])     # APE: black v.s. white (Black subpopulation), logit
            apes_prob_boot_bw_whites[t,i] = compute_APE(pe_prob_bw_boot[whites_dummy==0])      # APE: black v.s. white (White subpopulation), probit
            apes_logit_boot_bw_whites[t,i] = compute_APE(pe_logit_bw_boot[whites_dummy==0])     # APE: black v.s. white (White subpopultion), logit

            apes_prob_boot_hw_hisps[t,i] = compute_APE(pe_probit_hw_boot[hisp_dummy==1])       # APE: hisp v.s white (hisp sub), probit
            apes_logit_boot_hw_hisps[t,i] = compute_APE(pe_logit_hw_boot[hisp_dummy==1])      # APE: hisp v.s white (hisp sub), logit
            apes_prob_boot_hw_whites[t,i] = compute_APE(pe_probit_hw_boot[whites_dummy==0])      # APE: hisp v.s white (white sub), probit
            apes_logit_boot_hw_whites[t,i] = compute_APE(pe_logit_hw_boot[whites_dummy==0])     # APE: hist v.s white (white sub), logit

            



/Users/mathiaslindholmrasmussen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Semester 9/Advanced Microeconometrics/Adv_microeconometrics_assignments/project 3/w8_logit.py:14: RuntimeWarning: overflow encountered in exp
  Gz = 1. / (1. + np.exp(-z))
/Users/mathiaslindholmrasmussen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Semester 9/Advanced Microeconometrics/Adv_microeconometrics_assignments/project 3/w8_logit.py:14: RuntimeWarning: overflow encountered in exp
  Gz = 1. / (1. + np.exp(-z))
/Users/mathiaslindholmrasmussen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Semester 9/Advanced Microeconometrics/Adv_microeconometrics_assignments/project 3/w8_logit.py:14: RuntimeWarning: overflow encountered in exp
  Gz = 1. / (1. + np.exp(-z))


Bootstrap 1
Seperation failure
Variable sother in model x1
Seperation failure
Variable sother in model x2
Seperation failure
Variable sother in model x3
Seperation failure
Variable sother in model x4
Bootstrap 2
Seperation failure
Variable spop_3 in model x2
Seperation failure
Variable spop_3 in model x3
Seperation failure
Variable spop_3 in model x4
Bootstrap 3


/Users/mathiaslindholmrasmussen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Semester 9/Advanced Microeconometrics/Adv_microeconometrics_assignments/project 3/w8_logit.py:14: RuntimeWarning: overflow encountered in exp
  Gz = 1. / (1. + np.exp(-z))


Bootstrap 4


/Users/mathiaslindholmrasmussen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Semester 9/Advanced Microeconometrics/Adv_microeconometrics_assignments/project 3/w8_logit.py:14: RuntimeWarning: overflow encountered in exp
  Gz = 1. / (1. + np.exp(-z))


Bootstrap 5
Seperation failure
Variable spop_3 in model x2
Seperation failure
Variable spop_3 in model x3
Seperation failure
Variable spop_3 in model x4
Bootstrap 6
Seperation failure
Variable spop_3 in model x2
Seperation failure
Variable spop_3 in model x3
Seperation failure
Variable spop_3 in model x4
Bootstrap 7


... (2 outputs left out)

/Users/mathiaslindholmrasmussen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Semester 9/Advanced Microeconometrics/Adv_microeconometrics_assignments/project 3/w8_logit.py:14: RuntimeWarning: overflow encountered in exp
  Gz = 1. / (1. + np.exp(-z))


Seperation failure
Variable omajnonwhite in model x3
Seperation failure
Variable omajnonwhite in model x4
Bootstrap 9


/Users/mathiaslindholmrasmussen/Library/CloudStorage/OneDrive-UniversityofCopenhagen/Semester 9/Advanced Microeconometrics/Adv_microeconometrics_assignments/project 3/w8_logit.py:14: RuntimeWarning: overflow encountered in exp
  Gz = 1. / (1. + np.exp(-z))


Seperation failure
Variable omajnonwhite in model x3
Seperation failure
Variable omajnonwhite in model x4
Bootstrap 10
Seperation failure
Variable smale in model x2
Seperation failure
Variable smale in model x3
Seperation failure
Variable smale in model x4


In [177]:
def se_er(ape: np.array):
    se = np.nanstd(ape, ddof=1, axis=0)
    return se

In [ ]:
STD_BLACK_WHITE_PROBIT = se_er(apes_prob_boot_bw)
STD_BLACK_WHITE_LOGIT = se_er(apes_logit_boot_bw)
STD_HISP_WHITE_PROBIT = se_er(apes_prob_boot_hw)
STD_HISP_WHITE_LOGIT = se_er(apes_logit_boot_hw)

# Subpopulations
STD_BLACK_WHITE_PROBIT_BLACKS = se_er(apes_prob_boot_bw_blacks)
STD_BLACK_WHITE_LOGIT_BLACKS = se_er(apes_logit_boot_bw_blacks)
STD_BLACK_WHITE_PROBIT_WHITES = se_er(apes_prob_boot_bw_whites)
STD_BLACK_WHITE_LOGIT_WHITES = se_er(apes_logit_boot_bw_whites)

STD_HISP_WHITE_PROBIT_HISPS = se_er(apes_prob_boot_hw_hisps)
STD_HISP_WHITE_LOGIT_HISPS = se_er(apes_logit_boot_hw_hisps)
STD_HISP_WHITE_PROBIT_WHITES = se_er(apes_prob_boot_hw_whites)
STD_HISP_WHITE_LOGIT_WHITES = se_er(apes_logit_boot_hw_whites)


In [179]:
for h in range(4):
    APE_black_white_rows[h]['Probit s.e'] = STD_BLACK_WHITE_PROBIT[h]
    APE_black_white_rows[h]['Logit s.e'] = STD_BLACK_WHITE_LOGIT[h]
    APE_black_white_rows[h]['Probit s.e (black sub)'] = STD_BLACK_WHITE_PROBIT_BLACKS[h]
    APE_black_white_rows[h]['Logit s.e (black sub)'] = STD_BLACK_WHITE_LOGIT_BLACKS[h]
    APE_black_white_rows[h]['Probit s.e (white sub)'] = STD_BLACK_WHITE_PROBIT_WHITES[h]
    APE_black_white_rows[h]['Logit s.e (white sub)'] = STD_BLACK_WHITE_LOGIT_WHITES[h]

    APE_hisp_white_rows[h]['Probit s.e'] = STD_HISP_WHITE_PROBIT[h]
    APE_hisp_white_rows[h]['Logit s.e'] = STD_HISP_WHITE_LOGIT[h]
    APE_hisp_white_rows[h]['Probit s.e (hisp sub)'] = STD_HISP_WHITE_PROBIT_HISPS[h]
    APE_hisp_white_rows[h]['Logit s.e (hisp sub)'] = STD_HISP_WHITE_LOGIT_HISPS[h]
    APE_hisp_white_rows[h]['Probit s.e (white sub)'] = STD_HISP_WHITE_PROBIT_WHITES[h]
    APE_hisp_white_rows[h]['Logit s.e (white sub)'] = STD_HISP_WHITE_LOGIT_WHITES[h]


In [180]:
APE_black_white = pd.DataFrame(APE_black_white_rows).set_index("Model")
APE_hisp_white = pd.DataFrame(APE_hisp_white_rows).set_index("Model")

In [181]:
APE_black_white

,Probit APE,Logit APE,Probit APE (black sub),Logit APE (black sub),Probit APE (white sub),Logit APE (white sub),Probit s.e,Logit s.e,Probit s.e (black sub),Logit s.e (black sub),Probit s.e (white sub),Logit s.e (white sub)
Model,,,,,,,,,,,,
x1,0.003933,0.003934,0.003933,0.003934,0.003933,0.003934,0.003684,0.003688,0.003684,0.003688,0.003684,0.003688
x2,0.000926,0.001919,0.001144,0.002416,0.000807,0.001654,0.002051,0.002502,0.002459,0.003050,0.001804,0.002188
x3,0.001457,0.002447,0.001689,0.002892,0.001299,0.002166,0.002043,0.002709,0.002356,0.003228,0.001848,0.002446
x4,0.001700,0.001176,0.002013,0.001514,0.001479,0.000977,0.001763,0.001569,0.002201,0.002132,0.001549,0.001366


In [182]:
APE_hisp_white

,Probit APE,Logit APE,Probit APE (hisp sub),Logit APE (hisp sub),Probit APE (white sub),Logit APE (white sub),Probit s.e,Logit s.e,Probit s.e (hisp sub),Logit s.e (hisp sub),Probit s.e (white sub),Logit s.e (white sub)
Model,,,,,,,,,,,,
x1,0.012343,0.012352,0.012343,0.012352,0.012343,0.012352,0.007569,0.007586,0.007569,0.007586,0.007569,0.007586
x2,0.005805,0.005834,0.008671,0.009272,0.005102,0.005050,0.006486,0.007170,0.008685,0.009358,0.005654,0.006215
x3,0.006466,0.006673,0.009178,0.009882,0.005799,0.005922,0.007013,0.008596,0.007535,0.008830,0.006205,0.007526
x4,0.006018,0.005238,0.008841,0.008602,0.005294,0.004417,0.005579,0.006834,0.006547,0.008363,0.004582,0.005517


In [ ]:
### HISTOGRAM OF BOOTSTRAPS FOR PROBIT MODEL